# Comprehensibility Check

# Import packages

In [ ]:
## use kernel normgrading (Python 3.11.9)

## connecting with server
import pandas as pd
import numpy as np
import sys
import os
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from tqdm import tqdm
import re
import json
from datetime import datetime
import requests

In [ ]:
import openai
from openai._client import OpenAI
import dotenv

# Dataset

In [ ]:
path = os.getcwd()
#print("Path to current file:", path)
df = pd.read_csv(path+"/data/rotexamples_07072025.csv")

In [ ]:
df.head()

# Prompt

In [ ]:
## set model and temperature
def is_comprehensible(post, temp):
    keys = [
    "comprehensibility"
]

    system_prompt = """\
# System Prompt
You are an AI assistant trained to evaluate and annotate given social situation.
Rate each of the following questions on a scale of 1 (not at all) to 7 (very much).

# Criteria
"comprehensibility": "How easy is to understand the given social situation?",
"""

    json_schema = {
    "name": "annotation",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "comprehensibility": {
                "type": "integer",
                "description": "How easy is to understand the given social situation?",
                "enum": [1, 2, 3, 4, 5, 6, 7]
            }
        },
        "required": keys,
        "additionalProperties": False
    }
}

    response = openai.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": post
            }
        ],
        temperature=temp,
        max_tokens=2024,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
        response_format = {"type": "json_schema", "json_schema": json_schema}
      
    )

    output = response.choices[0].message.content
    # Parse the JSON output
    try:
        output = json.loads(output)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        output = None
    
    output_list = []
    if output is not None:
        output_list = [output[key] for key in keys if key in output]

    return output_list

# Calling LLM

In [ ]:
## LLM API Calls - key
import dotenv

lm = "openai" 

Openai_KEY_FILE_LOCATION = '/Users/sr/Github/NormGrading/openai_key.env'
deepseek_KEY_FILE_LOCATION = '/Users/sr/Github/NormGrading/deepseek_key.env'

if lm == "openai":
    dotenv.load_dotenv(Openai_KEY_FILE_LOCATION)
    openai.api_key = os.environ["OPENAI_API_KEY"]
    model = "gpt-4o-mini"
    temp=0
elif lm=="deepseek":
    dotenv.load_dotenv(deepseek_KEY_FILE_LOCATION)
    deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
    API_URL = "https://api.deepseek.com/v1/chat/completions"
    temp=0

In [ ]:
is_comprehensible("I am a student who is struggling with my studies. I feel overwhelmed and anxious about my exams.", temp)

In [ ]:
def get_annotations(lm, df, temp, message_column1, message_column2,  message_id_column, output):
    """
    Processes a dataframe to extract metadiscourse features for each post.
    Uses a progress bar to display the annotation progress.
    """

    print("Extracting style features...")

    # Initialize progress bar
    with tqdm(total=len(df), desc="Processing Posts", unit=message_column1) as pbar:
        for index, row in df.iterrows():
            result = list()
            if lm=="openai": ##running openai
                result = is_comprehensible(row[message_column1], temp)
                result2 = is_comprehensible(row[message_column2], temp)
        
            ## Adding message_id to output -- add first feature name below
            result.append(result2[0])
            result.append(row[message_id_column])
            result.append(row[message_column1])
            result.append(row[message_column2])
            output.append(result)  # Save output incrementally
            #print(result)
            pbar.update(1)  # Update progress bar

    print("annotation completed.")

In [19]:
output =  list()
get_annotations("openai", df, 0, "stimuli-male", "stimuli-female", "situation-short-id", output)

Extracting style features...


Processing Posts: 100%|██████████| 451/451 [09:58<00:00,  1.33s/stimuli-male]

annotation completed.


In [ ]:
cols = ["male_comprehensibility", "female_comprehensibility", "message_id", "stimuli-male", "stimuli-female"]
df_annotated = pd.DataFrame(output, columns=cols)

In [ ]:
##saving 
timestamp = datetime.now().strftime('%Y%m%d')
output_path = f"/Users/sr/Github/NormGrading/output/{lm}_{timestamp}.csv"
print(f"Saving output - {output_path}")
df_annotated.to_csv(output_path, index=False)

Saving output - /Users/sr/Github/NormGrading/output_openai_20250708.csv
